In [15]:
from langgraph.graph import StateGraph, START, END
from typing import TypedDict
import os
import time
from dotenv import load_dotenv
from langchain_groq import ChatGroq
from langgraph.checkpoint.memory import InMemorySaver  # inmemorysaver is a type of checkpointer which saves intermediate and final state values in RAM memory, used only for demos not production ready 

In [16]:
load_dotenv()

model = ChatGroq(
    api_key=os.getenv("GROQ_API_KEY"),
    model="llama-3.1-8b-instant"
)

In [17]:
class CrashState(TypedDict):
    input:str
    step1:str
    step2:str  
    step3:str

In [18]:
def step1(state:CrashState) -> CrashState:
    print("step 1 executed")
    return {"step1": "done", "input": state['input']}


def step2(state:CrashState) -> CrashState:
    print("step 2 hanging.. manually interupt")
    time.sleep(30)  # simulating a hanging step, you can manually interrupt the execution to test the crash handling
    return {"step2": "done"}

def step3(state:CrashState) -> CrashState:
    print("step 3 executed")
    return {"step3": "done"}
    

In [19]:
builder = StateGraph(CrashState)
builder.add_node('step1', step1)
builder.add_node('step2', step2)
builder.add_node('step3', step3)    

builder.add_edge(START, 'step1')
builder.add_edge('step1', 'step2')
builder.add_edge('step2', 'step3')  
builder.add_edge('step3', END)

checkpointer = InMemorySaver()  # create an instance of the checkpointer

graph = builder.compile(checkpointer=checkpointer)

In [20]:
try:
    print("Running graph: please manually interrupt during step 2 to simulate a crash")
    graph.invoke({"input":"start"}, config={"configurable":{"thread_id":"thread-1"}})
except KeyboardInterrupt:
    print("Kernel manually interrupted, simulating a crash...")

Running graph: please manually interrupt during step 2 to simulate a crash
step 1 executed
step 2 hanging.. manually interupt
Kernel manually interrupted, simulating a crash...


In [21]:
graph.get_state({"configurable":{"thread_id":"thread-1"}})

StateSnapshot(values={'input': 'start', 'step1': 'done'}, next=('step2',), config={'configurable': {'thread_id': 'thread-1', 'checkpoint_ns': '', 'checkpoint_id': '1f13ba4f-bb4a-61bc-8001-01ef3b674291'}}, metadata={'source': 'loop', 'step': 1, 'parents': {}}, created_at='2026-04-19T04:05:20.879036+00:00', parent_config={'configurable': {'thread_id': 'thread-1', 'checkpoint_ns': '', 'checkpoint_id': '1f13ba4f-bb47-64da-8000-77f81490bd90'}}, tasks=(PregelTask(id='41a15d22-be9d-ac4e-2233-1a2bbdb794b3', name='step2', path=('__pregel_pull', 'step2'), error=None, interrupts=(), state=None, result=None),), interrupts=())

In [22]:
list(graph.get_state_history({"configurable":{"thread_id":"thread-1"}}))

[StateSnapshot(values={'input': 'start', 'step1': 'done'}, next=('step2',), config={'configurable': {'thread_id': 'thread-1', 'checkpoint_ns': '', 'checkpoint_id': '1f13ba4f-bb4a-61bc-8001-01ef3b674291'}}, metadata={'source': 'loop', 'step': 1, 'parents': {}}, created_at='2026-04-19T04:05:20.879036+00:00', parent_config={'configurable': {'thread_id': 'thread-1', 'checkpoint_ns': '', 'checkpoint_id': '1f13ba4f-bb47-64da-8000-77f81490bd90'}}, tasks=(PregelTask(id='41a15d22-be9d-ac4e-2233-1a2bbdb794b3', name='step2', path=('__pregel_pull', 'step2'), error=None, interrupts=(), state=None, result=None),), interrupts=()),
 StateSnapshot(values={'input': 'start'}, next=('step1',), config={'configurable': {'thread_id': 'thread-1', 'checkpoint_ns': '', 'checkpoint_id': '1f13ba4f-bb47-64da-8000-77f81490bd90'}}, metadata={'source': 'loop', 'step': 0, 'parents': {}}, created_at='2026-04-19T04:05:20.877888+00:00', parent_config={'configurable': {'thread_id': 'thread-1', 'checkpoint_ns': '', 'checkp

In [23]:
# re-running to show fault tolerance is resumed
final_state = graph.invoke(None, config={"configurable":{"thread_id":"thread-1"}})
print("Final state after resuming from crash:", final_state)

step 2 hanging.. manually interupt
step 3 executed
Final state after resuming from crash: {'input': 'start', 'step1': 'done', 'step2': 'done', 'step3': 'done'}


In [24]:
graph.get_state({"configurable":{"thread_id":"thread-1"}})

StateSnapshot(values={'input': 'start', 'step1': 'done', 'step2': 'done', 'step3': 'done'}, next=(), config={'configurable': {'thread_id': 'thread-1', 'checkpoint_ns': '', 'checkpoint_id': '1f13ba51-307d-6898-8003-ed52984e9176'}}, metadata={'source': 'loop', 'step': 3, 'parents': {}}, created_at='2026-04-19T04:06:00.011975+00:00', parent_config={'configurable': {'thread_id': 'thread-1', 'checkpoint_ns': '', 'checkpoint_id': '1f13ba51-307b-620a-8002-f14a4d74f27d'}}, tasks=(), interrupts=())

In [25]:
list(graph.get_state_history({"configurable":{"thread_id":"thread-1"}}))

[StateSnapshot(values={'input': 'start', 'step1': 'done', 'step2': 'done', 'step3': 'done'}, next=(), config={'configurable': {'thread_id': 'thread-1', 'checkpoint_ns': '', 'checkpoint_id': '1f13ba51-307d-6898-8003-ed52984e9176'}}, metadata={'source': 'loop', 'step': 3, 'parents': {}}, created_at='2026-04-19T04:06:00.011975+00:00', parent_config={'configurable': {'thread_id': 'thread-1', 'checkpoint_ns': '', 'checkpoint_id': '1f13ba51-307b-620a-8002-f14a4d74f27d'}}, tasks=(), interrupts=()),
 StateSnapshot(values={'input': 'start', 'step1': 'done', 'step2': 'done'}, next=('step3',), config={'configurable': {'thread_id': 'thread-1', 'checkpoint_ns': '', 'checkpoint_id': '1f13ba51-307b-620a-8002-f14a4d74f27d'}}, metadata={'source': 'loop', 'step': 2, 'parents': {}}, created_at='2026-04-19T04:06:00.010980+00:00', parent_config={'configurable': {'thread_id': 'thread-1', 'checkpoint_ns': '', 'checkpoint_id': '1f13ba4f-bb4a-61bc-8001-01ef3b674291'}}, tasks=(PregelTask(id='6bf6192e-1bae-982a-